# Elasticsearch and OpenSearch

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/elasticsearch.ipynb)

Elasticsearch is a distributed search and analytics engine built on Apache Lucene. Documents are JSON, they live in indices split into shards and replicas, and nearly every query is answered from an inverted index rather than a table scan. OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2, so the mental model and most of the API carry over; the differences are licensing, plugins, and feature velocity.

For a data engineer the framing is: Elasticsearch is a serving layer for text search, log analytics, faceted aggregation, and low-latency filtered lookups. It is not a system of record and not a warehouse, and treating it as either is the fastest route to an expensive, fragile cluster.

The runnable cells build a tiny pure-Python search engine (analyzer, inverted index, BM25, bool queries, distributed aggregation and pagination) so each mechanism can be seen without a cluster; Query DSL and REST calls are shown as fenced snippets.

## What is Elasticsearch?

Elasticsearch stores JSON documents and answers two questions unusually fast: which documents contain these terms, and what are the counts and metrics grouped by these fields. It is built on Lucene segments, scales horizontally by splitting an index into shards, and tolerates node loss by keeping replicas on other nodes.

Nodes have roles. Master-eligible nodes hold the cluster state, data nodes hold shards, ingest nodes run pipelines, and coordinating-only nodes fan out requests and merge results. The cluster state (index metadata, mappings, allocation decisions) is replicated across master-eligible nodes, so a stable master quorum is part of availability.

```bash
curl -s 'localhost:9200/_cat/nodes?v&h=ip,node.role,heap.percent,cpu,load_1m'
curl -s 'localhost:9200/_cluster/health?wait_for_status=yellow&timeout=30s&pretty'
```

The path of a search request: the coordinating node fans the query out to one copy of every shard of the target index, each shard scores its local top hits, and the coordinator merges them into the global top `size`. That is why search latency tracks the slowest shard rather than the average. A point read is different: `GET /index/_doc/id` routes to a single shard and can read the document before a refresh has made it searchable.

Common mistake: reading `_cluster/health` green as "every node is fast". Green only means all shards are allocated. Yellow means some replicas are unassigned, which is normal on a single node but a real risk on a multi-node cluster.

## How are indices, shards, and replicas organized?

An index is a logical namespace split into primary shards, each an independent Lucene index on one node. Replicas copy a primary onto another node, serve reads, and take over on failure. Primary shard count is fixed at creation; replicas can change at any time.

```bash
curl -X PUT localhost:9200/orders-v1 -H 'Content-Type: application/json' -d '{
  "settings": {"number_of_shards": 6, "number_of_replicas": 1, "refresh_interval": "5s"}
}'

curl -X PUT localhost:9200/orders-v1/_settings -H 'Content-Type: application/json' \
  -d '{"index": {"number_of_replicas": 2}}'
curl -s 'localhost:9200/_cat/shards/orders-v1?v'
```

A document lands on a shard by a hash of its routing value, which defaults to `_id`: `shard = hash(routing) % number_of_primary_shards`. That is why the primary shard count cannot change in place - a different count would route the same id to a different shard. A custom `?routing=` value co-locates related documents on one shard, which is how a tenant-scoped query avoids a fan-out.

Reads go to one copy of the shard and the coordinating node picks a primary or replica using adaptive replica selection, which prefers the least loaded copy. Writes always go to the primary, which replicates to the in-sync replicas and only then acknowledges, so `?wait_for_active_shards` and `?refresh=wait_for` trade latency for stronger guarantees.

```bash
curl -s 'localhost:9200/_cat/shards/orders-v1?v&h=index,shard,prirep,state,docs,store,node'
curl -s 'localhost:9200/orders-v1/_count?routing=t-88'
```

Routing in miniature (Elasticsearch uses murmur3; any stable hash shows the effect): changing the primary count re-routes most ids, which is why it is fixed at creation, and a custom routing value co-locates a tenant on one shard.

In [ ]:
import hashlib


def shard_for(routing: str, number_of_primary_shards: int) -> int:
    digest = hashlib.md5(routing.encode()).digest()
    return int.from_bytes(digest[:4], "big") % number_of_primary_shards


ids = [f"order-{n}" for n in range(10_000)]
moved = sum(shard_for(i, 6) != shard_for(i, 7) for i in ids) / len(ids)
assert moved > 0.8                                   # 6 -> 7 shards would misroute most reads

tenant_docs = [("t-88", f"ticket-{n}") for n in range(50)]
by_id = {shard_for(doc_id, 6) for _, doc_id in tenant_docs}
by_tenant = {shard_for(tenant, 6) for tenant, _ in tenant_docs}
assert len(by_id) == 6 and len(by_tenant) == 1         # ?routing=t-88 -> one shard, no fan-out
print(f"ids re-routed by 6->7 shards: {moved:.0%}; shards hit by tenant t-88: {by_id and len(by_id)} vs {len(by_tenant)}")

## How do mappings work, and what is the difference between keyword and text?

The mapping is the schema: it declares each field's type and how it is indexed. Indexing an identifier as `text` and then filtering or aggregating on it is the classic mistake. `text` is analyzed into tokens and searched with relevance; `keyword` is stored verbatim for exact filtering, sorting, and aggregation.

```json
{
  "mappings": {
    "properties": {
      "order_id": { "type": "keyword" },
      "status": { "type": "keyword" },
      "description": { "type": "text", "analyzer": "english" },
      "total": { "type": "scaled_float", "scaling_factor": 100 },
      "created_at": { "type": "date" },
      "shipping": { "properties": {
        "city": { "type": "keyword" }, "location": { "type": "geo_point" } } }
    }
  }
}
```

When a field must serve both purposes, use a multi-field: `"description": {"type": "text", "fields": {"raw": {"type": "keyword"}}}` indexes the value twice. Other types worth naming: `long`, `double`, `scaled_float` for money, `date`, `boolean`, `ip`, `geo_point`, `nested` for arrays of objects that must not cross-match, and `flattened` for arbitrary key/value blobs.

Every mapped field controls up to three separate structures. The inverted index powers search. `doc_values` is a columnar copy on disk used for sorting, aggregations, and `search_after`. `norms` stores the field length that BM25 uses to normalize scores. A `keyword` field has doc values and no analyzer; a `text` field has the inverted index and norms, and no doc values.

```json
{
  "mappings": {
    "properties": {
      "sku":       { "type": "keyword" },
      "trace_id":  { "type": "keyword", "index": false },
      "log_level": { "type": "keyword", "norms": false, "doc_values": false },
      "body":      { "type": "text", "norms": false },
      "notes":     { "type": "text", "doc_values": false }
    }
  }
}
```

Use `index: false` for a field you only return from `_source` and never query. Use `doc_values: false` for a field you never sort or aggregate on. Use `norms: false` on filter-only `text` fields. For a `keyword` field that holds long values, `ignore_above` stops indexing strings over a limit.

Common mistake: disabling `_source` to save disk. `_source` is what `update`, `reindex`, and highlighting read, so dropping it breaks those features. Use `"_source": {"excludes": ["large_field"]}` to trim it instead.

## What are analyzers and how do they affect search?

An analyzer runs at index time and again at query time: character filters, then a tokenizer, then token filters. The analyzers to know are `standard` (Unicode word boundaries, lowercasing), `keyword` (the whole value as one token), and `whitespace` for special cases. A mismatch between index and query analysis is a top cause of "the document exists but search returns nothing".

Walk the chain with `_analyze`. A character filter rewrites the raw text (`html_strip`), the tokenizer splits it (`standard`, `whitespace`, `keyword`, `letter`), and token filters normalize the tokens (`lowercase`, `stop`, `asciifolding`, `porter_stem`, `synonym`). Index and search analysis must agree, so a custom analyzer sets `analyzer` for indexing and `search_analyzer` for queries when they differ.

```bash
curl -s 'localhost:9200/_analyze?pretty' -H 'Content-Type: application/json' -d '{
  "tokenizer": "standard",
  "filter": ["lowercase", "stop", "porter_stem"],
  "text": "The Quick Brown Foxes were running"
}'
```

A toy `english`-style analyzer (lowercase, stop words, a crude stemmer) is enough to show why `term` misses where `match` hits:

In [ ]:
import re

STOP_WORDS = {"the", "a", "an", "and", "or", "of", "to", "is", "are", "was", "were"}


def standard(text: str) -> list[str]:
    """Split on word boundaries and lowercase, like the standard analyzer."""
    return re.findall(r"\w+", text.lower())


def stem(token: str) -> str:
    """A deliberately tiny stemmer: running -> run, foxes -> fox, shoes -> shoe."""
    if token.endswith("ing") and len(token) > 5:
        token = token[:-3]
        return token[:-1] if token[-1] == token[-2] else token
    if token.endswith(("xes", "ches", "shes", "sses")):
        return token[:-2]
    if token.endswith("s") and not token.endswith("ss") and len(token) > 3:
        return token[:-1]
    return token


def english(text: str) -> list[str]:
    return [stem(t) for t in standard(text) if t not in STOP_WORDS]


assert standard("The 3 Quick Brown Foxes") == ["the", "3", "quick", "brown", "foxes"]
assert english("The Quick Brown Foxes were running") == ["quick", "brown", "fox", "run"]

indexed_tokens = set(english("Our runners kept running"))
match_tokens = english("Running")                     # match analyzes the query text
term_token = "Running"                                # term looks up the literal value
assert all(t in indexed_tokens for t in match_tokens)
assert term_token not in indexed_tokens
print("indexed:", sorted(indexed_tokens), "| match looks up", match_tokens, "| term looks up", [term_token])

## What are the pitfalls of dynamic mapping?

With dynamic mapping on, the first document defines the types, and a mixed field can silently produce a mapping you cannot change. A `long` cannot later store a decimal, and a field first seen as a number cannot accept text without reindexing.

```bash
# This fails once total is mapped as a number
curl -X POST localhost:9200/orders/_doc -H 'Content-Type: application/json' \
  -d '{"total": "unknown"}'
# mapper_parsing_exception: failed to parse field [total] of type [float]
```

Dynamic mapping decides types from the first value it sees. A JSON string becomes `text` plus a `.keyword` sub-field, an integer becomes `long`, a decimal becomes `float`, an object becomes a nested `properties` tree, and a date-looking string can become `date` when it matches `dynamic_date_formats`. Setting `"dynamic": "strict"` rejects an unknown field at index time, which surfaces schema drift as an error instead of a mapping explosion.

```json
{
  "mappings": {
    "dynamic": "strict",
    "dynamic_templates": [
      { "strings_as_keywords": { "match_mapping_type": "string",
          "mapping": { "type": "keyword" } } }
    ],
    "properties": {
      "@timestamp": { "type": "date" },
      "message": { "type": "text" }
    }
  }
}
```

## How does the inverted index work?

For each analyzed field, Lucene writes a term dictionary mapping every token to the documents containing it, with frequencies and positions. A query looks up terms and intersects or unions posting lists, which is why full-text search is fast regardless of corpus size. Separately, `doc_values` store values column-wise for sorting and aggregation, and `_source` stores the original JSON.

```text
Document 1: "red running shoes"    Document 2: "blue running jacket"    Document 3: "red jacket"
```

Worked example with the three documents above, analyzed by `standard`. The term dictionary holds `blue -> [2]`, `jacket -> [2,3]`, `red -> [1,3]`, `running -> [1,2]`, and `shoes -> [1]`. A `match` for `red jacket` becomes a boolean of two term lookups; the posting lists are intersected and the survivors are ranked by BM25. A `term` query for `Red` finds nothing, because the indexed token is lowercased to `red`.

Positions in the postings are what make `match_phrase` and proximity boosts possible, and the term frequencies are what BM25 scores. That is why `"index_options": "docs"` can save space on a field you only filter, at the cost of phrase queries.

Consequences: the inverted index only helps if the query uses indexed terms, which is why wildcards and leading-`*` regexes are slow (they scan terms) and why `text` fields cannot be sorted. Segment merges rewrite these structures, so indexing is I/O heavy and deletes are tombstones until a merge.

The engine below is reused by the next sections. It keeps per-field postings with positions (the inverted index), per-field lengths (norms), raw values for keyword and numeric fields (doc values), and scores with BM25 (`k1=1.2`, `b=0.75`, Lucene's defaults).

In [ ]:
import math
from collections import defaultdict


class MiniIndex:
    def __init__(self, mapping: dict[str, str], analyzer=standard):
        self.mapping = mapping                         # field -> "text" | "keyword" | "number"
        self.analyzer = analyzer
        self.docs: dict[str, dict] = {}                # _source
        self.postings = defaultdict(lambda: defaultdict(dict))   # field -> term -> {doc: [positions]}
        self.lengths = defaultdict(dict)               # field -> {doc: token count}

    def tokens(self, field: str, value) -> list[str]:
        return self.analyzer(value) if self.mapping[field] == "text" else [str(value)]

    def index(self, doc_id: str, source: dict) -> None:
        if doc_id in self.docs:                        # an update is delete + re-index
            self.delete(doc_id)
        self.docs[doc_id] = source
        for field, value in source.items():
            if self.mapping.get(field) in ("text", "keyword"):
                tokens = self.tokens(field, value)
                self.lengths[field][doc_id] = len(tokens)
                for position, token in enumerate(tokens):
                    self.postings[field][token].setdefault(doc_id, []).append(position)

    def delete(self, doc_id: str) -> None:
        self.docs.pop(doc_id)
        for field, terms in self.postings.items():
            for postings in terms.values():
                postings.pop(doc_id, None)
            self.lengths[field].pop(doc_id, None)

    def bm25(self, field: str, term: str, doc_id: str, k1: float = 1.2, b: float = 0.75) -> float:
        postings = self.postings[field].get(term, {})
        if doc_id not in postings:
            return 0.0
        n_docs, df = len(self.lengths[field]), len(postings)
        idf = math.log(1 + (n_docs - df + 0.5) / (df + 0.5))
        tf = len(postings[doc_id])
        avg_len = sum(self.lengths[field].values()) / n_docs
        norm = 1 - b + b * self.lengths[field][doc_id] / avg_len
        return idf * tf * (k1 + 1) / (tf + k1 * norm)

    # Each clause returns {doc_id: score}; filters just ignore the score.
    def match(self, field: str, text: str, operator: str = "or") -> dict[str, float]:
        terms = self.analyzer(text)
        sets = [set(self.postings[field].get(t, {})) for t in terms]
        hits = set.intersection(*sets) if operator == "and" else set.union(*sets)
        return {d: sum(self.bm25(field, t, d) for t in terms) for d in hits}

    def match_phrase(self, field: str, text: str) -> dict[str, float]:
        terms = self.analyzer(text)
        hits = {}
        for doc_id in set.intersection(*(set(self.postings[field].get(t, {})) for t in terms)):
            starts = self.postings[field][terms[0]][doc_id]
            if any(all(p + i in self.postings[field][t][doc_id] for i, t in enumerate(terms)) for p in starts):
                hits[doc_id] = sum(self.bm25(field, t, doc_id) for t in terms)
        return hits

    def term(self, field: str, value) -> dict[str, float]:
        return {d: 1.0 for d in self.postings[field].get(str(value), {})}   # no analysis

    def range(self, field: str, gte=None, lte=None) -> dict[str, float]:
        return {d: 1.0 for d, s in self.docs.items() if field in s
                and (gte is None or s[field] >= gte) and (lte is None or s[field] <= lte)}

    def bool(self, must=(), filter=(), should=(), must_not=(), minimum_should_match=0) -> list[tuple[str, float]]:
        candidates = set(self.docs)
        scores = defaultdict(float)
        for clause in must:
            candidates &= set(clause)
            for d, s in clause.items():
                scores[d] += s
        for clause in filter:                          # yes/no, no score: cacheable bitset
            candidates &= set(clause)
        for clause in must_not:
            candidates -= set(clause)
        should_hits = defaultdict(int)
        for clause in should:
            for d, s in clause.items():
                should_hits[d] += 1
                scores[d] += s
        candidates = {d for d in candidates if should_hits[d] >= minimum_should_match}
        return sorted(((d, round(scores[d], 4)) for d in candidates), key=lambda hit: (-hit[1], hit[0]))


idx = MiniIndex({"body": "text"})
for doc_id, body in {"1": "red running shoes", "2": "blue running jacket", "3": "red jacket"}.items():
    idx.index(doc_id, {"body": body})

term_dictionary = {term: sorted(p) for term, p in sorted(idx.postings["body"].items())}
assert term_dictionary == {"blue": ["2"], "jacket": ["2", "3"], "red": ["1", "3"],
                           "running": ["1", "2"], "shoes": ["1"]}
ranked = idx.bool(must=[idx.match("body", "red jacket")])
assert ranked[0][0] == "3"                              # both terms, shortest doc
assert idx.term("body", "Red") == {}                    # indexed token is "red"
assert set(idx.match_phrase("body", "running shoes")) == {"1"}   # positions keep adjacency
print("term dictionary:", term_dictionary)
print("match 'red jacket':", ranked)

## How do you index and update documents with the CRUD and bulk APIs?

Single-document calls are fine interactively; every pipeline should use `_bulk`, where each line is an action (`index`, `create`, `update`, `delete`) followed by the source, newline-delimited and without commas.

Indexing writes the document to an in-memory buffer and appends it to the translog for durability. A refresh builds a new Lucene segment from the buffer and opens a new searcher, which is what makes the document visible; the default `refresh_interval` is 1s. A flush persists the segments to disk and truncates the translog. The translog fsyncs per request by default (`index.translog.durability: request`); `async` with `sync_interval` trades a small window of loss for throughput.

```bash
curl -X PUT 'localhost:9200/orders/_doc/1001?refresh=wait_for' \
  -H 'Content-Type: application/json' -d '{"order_id":"1001","status":"paid"}'
curl -s 'localhost:9200/orders/_stats/refresh,flush,translog?pretty'
curl -s 'localhost:9200/orders/_cat/segments?v&h=index,shard,segment,docs.count,size'
```

Segment merging runs in the background. Lucene merges small segments into larger ones and reclaims deleted documents, which is why steady-state disk usage is lower than the raw write volume. Deletes and updates are tombstones until a merge. A `_bulk` call is newline-delimited JSON: an action line, then the source line, repeated, with no commas and a trailing newline.

```bash
curl -s -H 'Content-Type: application/x-ndjson' -X POST localhost:9200/_bulk --data-binary '
{"index":{"_index":"orders","_id":"1001"}}
{"order_id":"1001","status":"paid","total":42.50}
{"delete":{"_index":"orders","_id":"1000"}}
'
```

A `_bulk` body is NDJSON. Business-key `_id`s plus `doc_as_upsert` make a retry idempotent, and a `200` response still needs a per-item check. The simulated server below rejects some items with `429` on the first attempt:

In [ ]:
import json


def bulk_body(rows: list[dict]) -> str:
    lines = []
    for row in rows:
        lines.append(json.dumps({"update": {"_index": "orders", "_id": row["order_id"]}}))
        lines.append(json.dumps({"doc": row, "doc_as_upsert": True}))
    return "\n".join(lines) + "\n"                   # trailing newline is required


orders_idx = MiniIndex({"order_id": "keyword", "status": "keyword"})
attempts = defaultdict(int)


def fake_bulk(body: str) -> dict:
    lines = body.strip().split("\n")
    items = []
    for action_line, source_line in zip(lines[::2], lines[1::2]):
        doc_id = json.loads(action_line)["update"]["_id"]
        attempts[doc_id] += 1
        if attempts[doc_id] == 1 and int(doc_id[-1]) % 3 == 0:   # back-pressure on some items
            items.append({"update": {"_id": doc_id, "status": 429}})
            continue
        current = orders_idx.docs.get(doc_id, {})
        orders_idx.index(doc_id, {**current, **json.loads(source_line)["doc"]})
        items.append({"update": {"_id": doc_id, "status": 200}})
    return {"errors": any(i["update"]["status"] >= 300 for i in items), "items": items}


def load(rows: list[dict], max_retries: int = 3) -> int:
    pending, retries = rows, 0
    while pending:
        response = fake_bulk(bulk_body(pending))
        failed = {i["update"]["_id"] for i in response["items"] if i["update"]["status"] in (429, 503)}
        pending = [r for r in pending if r["order_id"] in failed]
        retries += bool(pending)
        if retries > max_retries:
            raise RuntimeError(f"giving up on {len(pending)} documents")
    return retries


rows = [{"order_id": f"o-{n}", "status": "paid"} for n in range(10)]
assert load(rows) == 1                                   # one retry round for the 429s
assert load(rows[:3]) == 0                               # replaying is a no-op overwrite
assert len(orders_idx.docs) == 10
print(bulk_body(rows[:1]), "documents:", len(orders_idx.docs))

With the real client the same loop is `elasticsearch.helpers.bulk`:

```python
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("https://es.internal:9200", api_key="...")
actions = ({"_op_type": "update", "_index": "orders", "_id": r["order_id"],
            "doc": r, "doc_as_upsert": True} for r in rows)
success, errors = bulk(es, actions, chunk_size=2000, max_retries=5,
                       initial_backoff=1, max_backoff=30, raise_on_error=False)
```

## How do you write search queries with match and term?

`match` analyzes the input and is the default for text; `term` matches one exact token and is for `keyword`, numbers, and booleans; `multi_match` searches several fields and combines their scores.

```json
{
  "query": {
    "bool": {
      "must": [
        { "multi_match": {
            "query": "wireless noise cancelling",
            "fields": ["title^3", "description", "brand^2"],
            "type": "best_fields"
        } }
      ],
      "filter": [
        { "term": { "status": "in_stock" } },
        { "range": { "price": { "gte": 50, "lte": 500 } } }
      ]
    }
  }
}
```

The `^3` boost makes title matches matter more, a first-order relevance lever before any ranking pipeline. `best_fields` scores the single best field, `most_fields` sums across fields, and `cross_fields` treats several fields as one, which is right for a first and last name searched together. Range queries support date math such as `"gte": "now-7d/d"`.

## How do bool queries combine clauses, and why filter instead of query?

A `bool` query has four clause types: `must` (scores and filters), `filter` (filters, no score, cacheable), `should` (optional, contributes score, and with `minimum_should_match` becomes required), and `must_not` (excludes).

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "description": "espresso machine" } } ],
      "filter": [
        { "term": { "category": "kitchen" } },
        { "terms": { "warehouse": ["DFW", "PHX"] } },
        { "range": { "stock": { "gt": 0 } } }
      ],
      "should": [
        { "term": { "tags": "coffee" } },
        { "term": { "tags": "sale" } }
      ],
      "minimum_should_match": 1,
      "must_not": [ { "term": { "status": "discontinued" } } ]
    }
  }
}
```

A query clause computes a BM25 `_score`, which is more expensive and depends on term statistics. A filter clause is a yes/no predicate: no score, and results cacheable in the filter cache. Elastic's own guidance is to filter anything that does not need to affect relevance.

In [ ]:
products = MiniIndex({"title": "text", "status": "keyword", "tags": "keyword", "price": "number"},
                     analyzer=english)
catalog = {
    "p1": {"title": "Espresso machine with milk frother", "status": "in_stock", "tags": "sale", "price": 249},
    "p2": {"title": "Manual espresso grinder", "status": "in_stock", "tags": "coffee", "price": 89},
    "p3": {"title": "Espresso machine", "status": "discontinued", "tags": "sale", "price": 199},
    "p4": {"title": "Drip coffee machine", "status": "in_stock", "tags": "coffee", "price": 45},
}
for doc_id, source in catalog.items():
    products.index(doc_id, source)

as_must = products.bool(must=[products.match("title", "espresso"), products.term("status", "in_stock")])
as_filter = products.bool(must=[products.match("title", "espresso")],
                          filter=[products.term("status", "in_stock"), products.range("price", gte=50)],
                          must_not=[products.term("status", "discontinued")])
assert [d for d, _ in as_must] == [d for d, _ in as_filter] == ["p2", "p1"]
assert as_filter[0][1] < as_must[0][1]                    # same docs; filter adds no score

boosted = products.bool(must=[products.match("title", "espresso")],
                        should=[products.term("tags", "sale")],
                        must_not=[products.term("status", "discontinued")])
assert boosted[0][0] == "p1"                              # should clause lifts p1 above p2
print("must  :", as_must)
print("filter:", as_filter)
print("should boost:", boosted)

## How do aggregations work in Elasticsearch?

Aggregations compute metrics (`avg`, `sum`, `min`, `max`, `percentiles`, `cardinality`) and buckets (`terms`, `date_histogram`, `range`, `histogram`). Buckets nest, and a terms bucket can carry sub-aggregations to build a report in one request.

```json
{
  "size": 0,
  "query": { "range": { "created_at": { "gte": "now-7d/d" } } },
  "aggs": {
    "by_day": {
      "date_histogram": { "field": "created_at", "calendar_interval": "day",
        "time_zone": "America/Chicago" },
      "aggs": { "revenue": { "sum": { "field": "total" } },
        "p95": { "percentiles": { "field": "total", "percents": [50, 95, 99] } } }
    },
    "top_customers": {
      "terms": { "field": "customer_id", "size": 10, "order": { "revenue": "desc" } },
      "aggs": { "revenue": { "sum": { "field": "total" } } }
    },
    "unique_customers": { "cardinality": { "field": "customer_id" } }
  }
}
```

A `terms` aggregation is distributed and approximate. Each shard returns its local top `size * shard_size` buckets, the coordinator merges them, and the response reports `doc_count_error_upper_bound` and `sum_other_doc_count`. On a high-cardinality field across many shards the edge buckets can be wrong. A `composite` aggregation pages every bucket exactly using `after_key`, at the cost of a sorted walk and one request per page.

```json
{
  "size": 0,
  "aggs": {
    "by_tenant": {
      "composite": {
        "size": 1000,
        "sources": [
          { "tenant": { "terms": { "field": "tenant_id" } } },
          { "day": { "date_histogram": { "field": "created_at", "calendar_interval": "day" } } }
        ]
      },
      "aggs": { "revenue": { "sum": { "field": "total" } } }
    }
  }
}
```

Why a distributed `terms` aggregation is approximate: each shard returns only its local top `shard_size` buckets, so a term that is second on every shard but never in a shard's top list can vanish from the merge.

In [ ]:
from collections import Counter

shards = [Counter(a=50, b=40, c=39), Counter(d=50, b=40, c=39), Counter(e=50, b=40, c=39)]


def terms_agg(shards: list[Counter], size: int, shard_size: int):
    merged, error_bound = Counter(), 0
    for shard in shards:
        top = shard.most_common(shard_size)
        merged.update(dict(top))
        if len(shard) > shard_size:                     # unseen buckets could be this big
            error_bound += top[-1][1]
    return merged.most_common(size), error_bound


exact = sum(shards, Counter()).most_common(2)
approx, error = terms_agg(shards, size=2, shard_size=2)
fixed, fixed_error = terms_agg(shards, size=2, shard_size=3)
assert exact == [("b", 120), ("c", 117)]
assert approx == [("b", 120), ("a", 50)] and error == 120   # c (117) was never returned
assert fixed == exact and fixed_error == 0
print("exact:", exact, "| shard_size=2:", approx, "error bound", error, "| shard_size=3:", fixed)

A `composite` aggregation instead walks every bucket in sorted key order and pages with `after_key`, which is exact:

In [ ]:
events = [("t2", "2026-10-01", 5), ("t1", "2026-10-02", 3), ("t1", "2026-10-01", 7),
          ("t3", "2026-10-01", 1), ("t2", "2026-10-02", 2)]
buckets = defaultdict(int)
for tenant, day, revenue in events:
    buckets[(tenant, day)] += revenue


def composite_page(after_key=None, size=2):
    keys = sorted(k for k in buckets if after_key is None or k > after_key)[:size]
    page = [{"key": {"tenant": t, "day": d}, "revenue": buckets[(t, d)]} for t, d in keys]
    return page, (keys[-1] if len(keys) == size else None)


pages, after = [], None
while True:
    page, after = composite_page(after)
    pages.append(page)
    if after is None:
        break
assert sum(len(p) for p in pages) == len(buckets) == 5
assert pages[0][0] == {"key": {"tenant": "t1", "day": "2026-10-01"}, "revenue": 7}
print([len(p) for p in pages], "buckets per page")

`cardinality` uses HyperLogLog++ and is approximate. `precision_threshold` (default 3000, max 40000) sets the point where counts become near-exact, and raising it costs memory. `date_histogram` needs a `time_zone` when buckets must follow a local calendar day.

Common mistake: aggregating on a `text` field. Terms, sort, and cardinality need doc values, a `text` field has none, and enabling `"fielddata": true` loads the field into the JVM heap. That is the classic cause of a search node running out of memory. Aggregate on the `keyword` field or a `.keyword` sub-field instead.

```bash
curl -s 'localhost:9200/_cat/fielddata?v&h=node,field,size'
curl -s 'localhost:9200/_nodes/stats/indices/fielddata?pretty'
```

## How does pagination work, and what are the from and size limits?

Deep pagination with `from`/`size` is capped by `index.max_result_window` (default 10,000) because every shard fetches and sorts `from + size` hits. Beyond that, use `search_after` with a stable sort, pinned to a point-in-time (PIT) so the sort order cannot shift as documents are indexed.

In [ ]:
import random

rng = random.Random(42)
docs = [{"_id": f"d{n:03d}", "created_at": f"2026-10-0{rng.randint(1, 3)}"} for n in range(250)]
sort_key = lambda d: (-int(d["created_at"].replace("-", "")), d["_id"])   # desc date, then _id tiebreak


def search_after(after=None, size=100):
    ordered = sorted(docs, key=sort_key)
    if after is not None:
        ordered = [d for d in ordered if sort_key(d) > after]
    page = ordered[:size]
    return page, (sort_key(page[-1]) if page else None)


seen, after = [], None
while True:
    page, after = search_after(after)
    if not page:
        break
    seen.extend(d["_id"] for d in page)
assert len(seen) == len(set(seen)) == 250                # no repeats, no gaps

shards, frm, size = 20, 9_000, 100
assert shards * (frm + size) == 182_000                  # hits sorted for one deep from/size page
assert shards * size == 2_000                            # the same page with search_after
print("pages walked without duplicates:", len(seen), "docs")

## What are index templates and how do you use them?

A template applies settings, mappings, and aliases to every index whose name matches a pattern. Composable and component templates let shared settings be defined once and overridden per dataset, which is how a logging platform creates correctly mapped daily indices without humans in the loop.

```bash
curl -X PUT localhost:9200/_index_template/logs-app -H 'Content-Type: application/json' -d '{
  "index_patterns": ["logs-app-*"],
  "priority": 200,
  "template": {
    "settings": {
      "number_of_shards": 3, "number_of_replicas": 1, "refresh_interval": "10s",
      "index.lifecycle.name": "logs-ilm",
      "index.default_pipeline": "logs-parse"
    },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}'
```

## How do aliases and reindex work?

An alias is a second name for one or more indices. Writing through an alias with `is_write_index` and searching through an alias that fans out over several indices is the standard pattern: applications never hard-code an index name, so a rebuild is invisible.

```bash
curl -X POST localhost:9200/_reindex?wait_for_completion=false -H 'Content-Type: application/json' -d '{
  "source": { "index": "orders-v1" },
  "dest": { "index": "orders-v2", "op_type": "create" }
}'

curl -X POST localhost:9200/_aliases -H 'Content-Type: application/json' -d '{
  "actions": [
    { "remove": { "index": "orders-v1", "alias": "orders" } },
    { "add": { "index": "orders-v2", "alias": "orders", "is_write_index": true } }
  ]
}'
```

`_reindex` copies documents and applies the destination mapping, so it is also the migration path after a mapping change - a field's type cannot be changed in place. It is not incremental by itself; for live data, pause writes or dual-write while the reindex catches up, then flip. `slices` parallelizes a large reindex.

## How does index lifecycle management work?

ILM moves an index through hot, warm, cold, frozen, and delete phases based on age or size. Rollover creates a new backing index at a threshold, and each phase can change replicas, force-merge, move to cheaper nodes, or shrink.

```json
{
  "policy": {
    "phases": {
      "hot": { "actions": {
        "rollover": { "max_primary_shard_size": "50gb", "max_age": "1d" },
        "set_priority": { "priority": 100 } } },
      "warm": { "min_age": "2d", "actions": {
        "forcemerge": { "max_num_segments": 1 }, "set_priority": { "priority": 50 } } },
      "cold": { "min_age": "30d", "actions": {
        "readonly": {}, "searchable_snapshot": {}
      } },
      "delete": { "min_age": "365d", "actions": { "delete": {} } }
    }
  }
}
```

Attach it with `"index.lifecycle.name": "logs-ilm"` in the template and write through a rollover alias. Hot nodes need fast NVMe and CPU, warm nodes cheaper disk, and cold/frozen tiers keep data searchable from object storage. `searchable_snapshot` is the biggest cost lever for log retention because it removes the local copy.

Common mistake: assuming `min_age: "30d"` on the delete phase means 30 days from index creation. For an index that rolls over, ages after the hot phase are measured from the rollover time. Use `_ilm/explain` to confirm the age and the phase an index is actually in.

## How do you ingest data with Beats, Logstash, and Kafka Connect?

There are four common paths: Beats agents ship logs and metrics; Logstash parses, enriches, and fans out; Kafka Connect sinks records from Kafka; and small services call the bulk API directly. For a data platform, Kafka plus a sink is usually most robust because Kafka absorbs spikes and replay handles index rebuilds.

```yaml
input:
  kafka:
    bootstrap_servers: "broker1:9092,broker2:9092"
    topics: ["logs.app"]
    codec: json
filter:
  date:
    match: ["ts", "ISO8601"]
    target: "@timestamp"
output:
  elasticsearch:
    hosts: ["https://es.internal:9200"]
    index: "logs-app-%{+YYYY.MM.dd}"
    ilm_enabled: true
    manage_template: true
```

```text
Beats --> Logstash --> Elasticsearch
Kafka --> Kafka Connect (Elasticsearch sink, bulk, retries) --> Elasticsearch
CDC (Debezium) --> Kafka --> sink --> Elasticsearch
```

Kafka Connect's sink batches, retries, and supports exactly-once with a dead-letter queue for rejected records. Use a business key as the document `_id` (for example `order_id`) so replays overwrite rather than duplicate, and always validate an ingest pipeline with `_ingest/pipeline/_simulate` before pointing production traffic at it.

## How does OpenSearch differ from Elasticsearch?

OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2 plus OpenSearch Dashboards. Core concepts, mapping types, the DSL, and the bulk API are compatible enough that knowledge transfers; the differences are at the edges.

| Aspect | Elasticsearch | OpenSearch |
|---|---|---|
| License | Elastic License 2.0 / SSPL | Apache License 2.0 |
| Origin | Elastic / Lucene | AWS-led fork of 7.10.2 |
| Vector search | dense_vector, kNN, ELSER | kNN, neural search plugin |
| SQL | SQL, ESQL | SQL and PPL |
| Alerting | Kibana alerting | Alerting plugin, anomaly detection |
| Dashboards | Kibana | OpenSearch Dashboards |
| Managed | Elastic Cloud, AWS (limited) | AWS OpenSearch Service, self-hosted |

OpenSearch wins when the license, an AWS-native managed service, or PPL/SQL querying matters; Elasticsearch wins on the latest relevance and vector features. Both keep the same operational shape - shards, mappings, templates, ILM (ISM in OpenSearch), and snapshots - so a migration is mostly reindexing and re-creating dashboards and alerts, not re-learning search.

## How do you size a cluster and choose shard count?

Sizing starts from data volume, retention, and query shape. Estimate raw JSON size, multiply for `_source` plus indexes, add replicas and headroom, and divide into shards of 10-50 GB. Memory is dominated by the JVM heap (cap around 31 GB to keep compressed ordinary object pointers) and by the filesystem cache, which should hold the hot working set.

```text
1 TB/day raw JSON, 30-day retention
  stored/day          ~ 1 TB * 0.40 compression = 400 GB primary
  with one replica    ~ 800 GB/day
  30 days             ~ 24 TB per replica set (before cold/frozen offload)
  shards at 40 GB     ~ 10 primary shards/day, rolled over daily
```

The heap is the first constraint. Set the JVM heap to at most half of RAM and at most about 31 GB so compressed ordinary object pointers stay on; the rest of RAM is filesystem cache for Lucene segments. A rough planning figure is 20 shards per GB of heap, but the real limit is the hot working set: if the recently searched segments do not fit in page cache, every search becomes a disk read. Count shards including replicas, and remember each shard carries segments, a translog, and cluster-state metadata.

```bash
curl -s 'localhost:9200/_cat/allocation?v&h=node,shards,disk.used_percent,disk.avail'
curl -s 'localhost:9200/_nodes/stats/jvm,fs,indices/segments?pretty'
```

In [ ]:
import math

RAW_GB_PER_DAY = 1000
COMPRESSION = 0.40
TARGET_SHARD_GB = 40
REPLICAS = 1
HEAP_GB_PER_NODE = 31
MAX_SHARDS_PER_HEAP_GB = 20

primary_gb_per_day = RAW_GB_PER_DAY * COMPRESSION
shards_per_day = math.ceil(primary_gb_per_day / TARGET_SHARD_GB)
shard_budget_per_node = HEAP_GB_PER_NODE * MAX_SHARDS_PER_HEAP_GB
total_shards_at_30d = shards_per_day * 30 * (1 + REPLICAS)

print(f"primary GB/day: {primary_gb_per_day:.0f}")
print(f"shards/day at {TARGET_SHARD_GB} GB: {shards_per_day}")
print(f"shards after 30 days with {REPLICAS} replica(s): {total_shards_at_30d}")
print(f"shard budget per 31 GB heap node: {shard_budget_per_node}")

assert primary_gb_per_day == 400.0
assert shards_per_day == 10
assert total_shards_at_30d == 600
assert shard_budget_per_node == 620

## How do you tune indexing and refresh?

The write path is buffer, translog, refresh, flush, merge. `refresh_interval` controls visibility, not durability: 30s or 60s instead of 1s cuts segment count and search overhead during a bulk load. Set `"refresh": false` on bulk requests during a backfill and refresh once at the end. Durability comes from the translog, so `index.translog.durability: async` risks only the sync window, not the whole load.

```bash
curl -X PUT localhost:9200/orders/_settings -H 'Content-Type: application/json' \
  -d '{"index": {"refresh_interval": "-1"}}'
curl -s -H 'Content-Type: application/x-ndjson' -X POST \
  'localhost:9200/_bulk?refresh=false' --data-binary @bulk.ndjson
curl -X POST 'localhost:9200/orders/_refresh'
```

Bulk sizing: 1,000 to 5,000 documents or about 5-15 MB per request is the usual sweet spot. Too small wastes round trips and creates many segments; too large risks memory pressure, long retries, and a single failure losing a big batch. Send bulk requests to several nodes rather than one. Replicas double the write work, so a backfill can temporarily set `number_of_replicas: 0` and restore it after. Merge tuning matters too: `index.merge.scheduler.max_thread_count: 1` reduces interference on spinning or network disks, and a larger `indices.memory.index_buffer_size` builds bigger segments before merge.

## How do you profile and tune search performance?

Start with the slow log and the profile API, not guesswork. Search is slow for recurring reasons: a wildcard or script query, deep `from`/`size`, a high-cardinality `terms` aggregation, a field without `doc_values`, or a query that fans out to every shard.

```bash
curl -X PUT localhost:9200/orders/_settings -H 'Content-Type: application/json' \
  -d '{"index.search.slowlog.threshold.query.warn": "2s"}'

curl -X POST 'localhost:9200/orders/_search?pretty' -H 'Content-Type: application/json' \
  -d '{"profile": true, "query": {"bool": {"must": [{"match": {"description": "espresso"}}]}}}'

curl -s 'localhost:9200/_nodes/stats/indices/search,indices/query_cache?pretty'
```

Fixes by payoff: move non-scoring constraints into `filter`, replace wildcards with an analyzed field or `keyword` prefix, add `track_total_hits: false`, reduce `size` and use `search_after`, pre-aggregate with transforms or rollups, and raise `refresh_interval` during heavy ingestion. Force-merging read-only indices reduces segment count and improves latency, but is expensive and only safe on indices no longer written.

For aggregations, prefer `keyword` fields with doc values, keep the terms `size` small, use `composite` when you need every bucket, and set `precision_threshold` on `cardinality` deliberately. Never paginate deeply with `from`/`size`; use PIT plus `search_after`. The request cache helps repeated identical `size: 0` searches and the query cache helps repeated filters, so watch their eviction counts: evictions mean the working set does not fit.

In [ ]:
import math

SHARDS = 30
TERMS_SIZE = 100
SHARD_SIZE = int(TERMS_SIZE * 1.5) + 10
BYTES_PER_BUCKET = 96
DOCS = 2_000_000_000

coordinator_buckets = SHARDS * SHARD_SIZE
coordinator_bytes = coordinator_buckets * BYTES_PER_BUCKET
docs_per_shard = DOCS / SHARDS

print(f"candidate buckets per shard: {SHARD_SIZE}")
print(f"buckets merged at the coordinator: {coordinator_buckets:,}")
print(f"coordinator bucket memory: {coordinator_bytes / 1024:.1f} KiB")
print(f"documents per shard: {docs_per_shard:,.0f}")

assert SHARD_SIZE == 160
assert coordinator_buckets == 4800
assert coordinator_bytes == 460_800
assert round(docs_per_shard, 2) == 66666666.67

## How do snapshots and backups work?

A snapshot is a point-in-time copy of indices and cluster state into a repository such as S3, GCS, Azure Blob, or a shared filesystem. Snapshots are incremental, so unchanged segments are referenced rather than recopied, which makes frequent snapshots cheap. It is the only sanctioned backup mechanism; copying data directories is not.

```bash
curl -X PUT localhost:9200/_snapshot/s3-backups -H 'Content-Type: application/json' \
  -d '{"type": "s3", "settings": {"bucket": "acme-es-backups", "base_path": "prod"}}'

curl -X PUT 'localhost:9200/_snapshot/s3-backups/snap-2026-10-04?wait_for_completion=true' \
  -H 'Content-Type: application/json' -d '{
    "indices": "orders,logs-app-*", "include_global_state": true, "ignore_unavailable": true
  }'

curl -X POST localhost:9200/_snapshot/s3-backups/snap-2026-10-04/_restore \
  -H 'Content-Type: application/json' \
  -d '{"indices": "orders", "rename_pattern": "orders", "rename_replacement": "orders_restored"}'
```

## How do you secure an Elasticsearch cluster?

Security has four layers: transport and HTTP TLS, authentication, authorization (roles plus document and field-level access), and audit logging. Exposing an unauthenticated cluster to a network is a breach waiting for a scanner.

```yaml
xpack.security.enabled: true
xpack.security.transport.ssl.enabled: true
xpack.security.transport.ssl.verification_mode: certificate
xpack.security.transport.ssl.keystore.path: certs/elastic-certificates.p12
xpack.security.http.ssl.enabled: true
xpack.security.http.ssl.keystore.path: certs/http.p12
```

```bash
curl -X POST localhost:9200/_security/role/logs_reader -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["logs-app-*"], "privileges": ["read"]}]}'

curl -X POST localhost:9200/_security/role/tenant_t88 -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["tickets-*"], "privileges": ["read"],
        "query": {"term": {"tenant_id": "t-88"}}}]}'
```

## How do you design a log analytics architecture?

A log platform is a pipeline with a hot serving tier and a cold archive tier: producers write JSON with a timestamp and service identifier, shippers buffer and compress, a processing tier parses and enriches, and the index tier serves a bounded window while older data lives in object storage.

```text
apps --> Fluent Bit/Filebeat --> Kafka --> Logstash/Connect --> Elasticsearch
                                      |                              |
                                      |                              +-- ILM hot -> warm -> frozen -> delete
                                      +--> S3 archive (Parquet) --> Trino/Athena
```

Decisions that matter: normalize timestamp and service in the template so every dashboard works across indices; keep identifiers as `keyword` and free text as `text`; use `dynamic: strict` so a new field cannot blow up the mapping; set `refresh_interval` to 10-30s because dashboards do not need sub-second freshness; and sample or drop debug logs at the shipper. The Parquet archive is not optional - search clusters are expensive per byte, and Parquet answers compliance retention at a fraction of the cost.

A data stream is the modern shape for logs. It is a named write target backed by hidden, rolled-over indices, ILM handles rollover and retention, and the name always points at the writable backing index. The mapping is fixed by the template, so a new incompatible field fails at index time instead of silently changing the stream. A search over the stream fans out across the backing indices and their shards.

```bash
curl -X PUT localhost:9200/_data_stream/logs-app -H 'Content-Type: application/json' \
  -d '{"template": "logs-app-template"}'
curl -s 'localhost:9200/_data_stream/logs-app?pretty'
curl -s 'localhost:9200/_cat/indices/.ds-logs-app-*?v&h=index,docs.count,store.size'
```

## How does search relevance work?

Relevance is the BM25 `_score`, which rewards term frequency, penalizes common terms, and normalizes by document length. The practical levers in order: get the analyzer right, use `multi_match` with field boosts, add `should` clauses for recency or popularity, use `function_score` for business signals, and only then consider learning-to-rank.

```json
{
  "query": {
    "function_score": {
      "query": { "multi_match": { "query": "espresso machine", "fields": ["title^3", "description"] } },
      "functions": [
        { "filter": { "term": { "in_stock": true } }, "weight": 2 },
        { "gauss": { "published_at": { "origin": "now", "scale": "30d", "decay": 0.5 } } }
      ],
      "score_mode": "sum",
      "boost_mode": "multiply"
    }
  }
}
```

`explain` and judgments are how you tune relevance without guessing. `"explain": true` returns the BM25 term, idf, tf, and norm components for one hit. A judgment set pairs a query with known-good document ids, and nDCG or MRR scores a change before it ships. A `_score` is comparable only within a single query, never across queries.

```bash
curl -s 'localhost:9200/products/_search?pretty&explain=true' -H 'Content-Type: application/json' \
  -d '{"query": {"match": {"title": "espresso machine"}}, "size": 1}'
```

## Elasticsearch vs ClickHouse vs Parquet and Trino?

All three answer analytical questions at different points on the latency/retention/operability curve. The choice follows the query shape: text search and faceted navigation go to Elasticsearch, high-concurrency columnar aggregation to ClickHouse, and cheap large-scale retention to Parquet queried by Trino.

| Dimension | Elasticsearch / OpenSearch | ClickHouse | Parquet + Trino |
|---|---|---|---|
| Storage | inverted index + doc values | columnar MergeTree parts | columnar immutable files |
| Best at | text search, facets, logs | sub-second aggregations | large scans, open data |
| Full-text | excellent, analyzers, relevance | basic token indexes | none natively |
| Updates | per document, near-real-time | async mutations | rewrite or delete files |
| Concurrency | high for search, agg-heavy costly | very high for aggregations | high via a cluster |
| Storage cost | highest | medium | lowest |
| Retention | ILM tiers, expensive per byte | local disks or S3 tiered | effectively unlimited |
| SQL | SQL/ESQL, limited joins | strong SQL, limited joins | full SQL and joins |
| Ops | JVM cluster, shard tuning | self-managed tuning | stateless query layer + metastore |

A defensible architecture uses all three: Kafka to Parquet for cheap immutable history, Trino or Spark for batch analytics over it, ClickHouse or a warehouse for interactive dashboards, and Elasticsearch for search, log exploration, and faceted filtering. Choosing one for everything is how teams end up with a search cluster doing warehousing.

## Real world use cases for Elasticsearch

### 1. Log analytics and observability

Business context: 4,000 services emit 1.5 TB of logs/day; engineers search the last 7 days in under a second and compliance needs 12 months. Design: Filebeat to Kafka, Connect for parsing into a strict template, daily rollover indices with ILM, and a Parquet archive in S3.

```json
{
  "index_patterns": ["logs-app-*"],
  "template": {
    "settings": { "refresh_interval": "10s", "index.lifecycle.name": "logs-ilm" },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}
```

Why Elasticsearch: analyzed text search plus `terms` and `date_histogram` answer both "find this error" and "how many per service per hour", and ILM moves older data to cheaper tiers.

### 2. E-commerce search with faceted navigation

Business context: 8M SKUs, 12,000 searches/s, filters on brand, price, and rating, with continuous catalog updates. Design: `text` for title and description, `keyword` for brand, `scaled_float` for price, Kafka Connect with the SKU as `_id`, and an alias for reindex.

```json
{
  "size": 20,
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "noise cancelling headphones",
        "fields": ["title^3", "brand^2", "description"] } } ],
      "filter": [ { "term": { "in_stock": true } },
        { "range": { "price": { "gte": 50, "lte": 400 } } } ]
    }
  },
  "aggs": {
    "brands": { "terms": { "field": "brand", "size": 20 } },
    "avg_rating": { "avg": { "field": "rating" } }
  }
}
```

Why Elasticsearch: one request returns ranked hits and the filter counts the UI needs, and boosting plus `function_score` lets merchandising influence order.

### 3. Security information and event management

Business context: a security team correlates events from 60 sources and needs continuous detection plus forensic search over 13 months. Design: normalized events, a write alias with rollover, ILM with a frozen tier, scheduled detection searches, and immutable snapshots.

```text
detection rule, every 5 minutes
  search:   event.category:authentication AND event.outcome:failure
  group by: source.ip
  trigger:  count > 20 within 5 minutes
```

Why Elasticsearch: high-cardinality keyword filtering plus aggregations make correlation practical, and each hit keeps its `_source` for investigation.

### 4. Product catalog sync from a lakehouse

Business context: the warehouse (Iceberg on S3, queried by Trino) is the system of record; the search index must reflect changes within minutes and survive a rebuild without downtime. Design: a curated Parquet export, a bulk-index job into a versioned index, and an alias flip.

```python
import pyarrow.dataset as ds
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk

es = Elasticsearch("https://es.internal:9200", api_key="...")


def documents(path: str, batch_size: int = 5000):
    for batch in ds.dataset(path, format="parquet").to_batches(batch_size=batch_size):
        for row in batch.to_pylist():
            yield {"_index": "products-v3", "_id": row["sku"], "_source": row}


for ok, item in streaming_bulk(es, documents("s3://lake/curated/products/"), chunk_size=2000, max_retries=5):
    ...
# then flip the alias: remove products -> products-v2, add products -> products-v3
```

Why Elasticsearch: bulk indexing with business-key ids is idempotent, the alias makes the swap invisible, and the lakehouse stays the single source of truth.

### 5. Multi-tenant customer support search

Business context: a SaaS helpdesk holds 900M tickets across 2,000 tenants, every query is tenant-scoped, and one tenant is 30% of the data. Design: a shared index with `tenant_id` as `keyword`, document-level security, routing by `tenant_id`, and per-contract ILM retention.

```json
{
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "refund not received",
        "fields": ["subject^2", "body"] } } ],
      "filter": [
        { "term": { "tenant_id": "t-88" } },
        { "range": { "created_at": { "gte": "now-90d/d" } } }
      ]
    }
  },
  "highlight": { "fields": { "body": {} } }
}
```

```bash
curl -X POST 'localhost:9200/tickets/_doc/9001?routing=t-88' -H 'Content-Type: application/json' \
  -d '{"tenant_id":"t-88","subject":"Refund missing","body":"I have not received my refund."}'
```

Why Elasticsearch: document-level security plus routing gives tenant isolation and locality, highlight returns the snippet the UI needs, and per-tenant retention is an ILM concern.

## Elasticsearch exercises

The numbers are estimates, not promises. Real results depend on compression, cardinality, mapping, and cluster shape; the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Choose shard count from index size and growth

**Task.** An index holds 500 GB of primary data today and grows 8% per month. Target 40 GB per shard and plan for a 12-month horizon. Choose the number of primary shards.

**Hint.** Project the size with `current * (1 + growth) ** months`, then take `ceil(projected / target)`.

**Expected result.** About 1,259.1 GB projected, which rounds up to 32 primary shards.

In [ ]:
import math

CURRENT_GB = 500
MONTHLY_GROWTH = 0.08
HORIZON_MONTHS = 12
TARGET_SHARD_GB = 40

projected_gb = CURRENT_GB * (1 + MONTHLY_GROWTH) ** HORIZON_MONTHS
shards = math.ceil(projected_gb / TARGET_SHARD_GB)

print(f"projected primary data: {projected_gb:,.1f} GB")
print(f"primary shards at {TARGET_SHARD_GB} GB: {shards}")

assert round(projected_gb, 1) == 1259.1
assert shards == 32

### Try it 2. Compute node and storage needs from replicas and overhead

**Task.** A cluster holds 12 TB of primary data with one replica and keeps 30% of each disk free. Each node provides 8 TB of usable storage. Compute the logical size, the raw storage needed, and the node count.

**Hint.** `logical = primary * (1 + replicas)`; `raw = logical / (1 - free_fraction)`; `nodes = ceil(raw / usable_per_node)`.

**Expected result.** 24 TB logical, about 34.29 TB raw, and 5 nodes.

In [ ]:
import math

PRIMARY_TB = 12
REPLICAS = 1
FREE_SPACE = 0.30
USABLE_TB_PER_NODE = 8

logical_tb = PRIMARY_TB * (1 + REPLICAS)
raw_tb = logical_tb / (1 - FREE_SPACE)
nodes = math.ceil(raw_tb / USABLE_TB_PER_NODE)

print(f"logical with {REPLICAS} replica(s): {logical_tb:.0f} TB")
print(f"raw storage at {FREE_SPACE:.0%} free: {raw_tb:.2f} TB")
print(f"data nodes needed: {nodes}")

assert logical_tb == 24
assert round(raw_tb, 2) == 34.29
assert nodes == 5

### Try it 3. Decide keyword versus text from query patterns

**Task.** Choose a field type for four fields: `order_id` (filtered and sorted), `title` (full-text searched only), `status` (filtered only), and `description` (searched for text and aggregated as a whole value).

**Hint.** Full text needs `text`; exact filter or sort needs `keyword`; both at once needs a `text` field with a `keyword` sub-field.

**Expected result.** `keyword`, `text`, `keyword`, `text with keyword sub-field`.

In [ ]:
def choose_type(full_text, exact_use):
    if full_text and exact_use:
        return "text with keyword sub-field"
    if full_text:
        return "text"
    return "keyword"

fields = [
    ("order_id", False, True),
    ("title", True, False),
    ("status", False, True),
    ("description", True, True),
]
result = {name: choose_type(full_text, exact_use) for name, full_text, exact_use in fields}
for name, mapping in result.items():
    print(f"{name}: {mapping}")

assert result["order_id"] == "keyword"
assert result["title"] == "text"
assert result["status"] == "keyword"
assert result["description"] == "text with keyword sub-field"

### Try it 4. Detect oversharding from shard count and cluster state

**Task.** A 5-node cluster holds 96,000 GB across 120,000 shards. Flag the cluster as oversharded if the average shard is under 10 GB or any node holds more than 1,000 shards.

**Hint.** `average_gb = total_gb / shards`; `per_node = shards / nodes`.

**Expected result.** 0.8 GB per shard and 24,000 shards per node, so the cluster is oversharded.

In [ ]:
DATA_NODES = 5
SHARDS = 120_000
TOTAL_GB = 96_000
MIN_SHARD_GB = 10
MAX_SHARDS_PER_NODE = 1_000

average_gb = TOTAL_GB / SHARDS
per_node = SHARDS / DATA_NODES
oversharded = average_gb < MIN_SHARD_GB or per_node > MAX_SHARDS_PER_NODE

print(f"average shard: {average_gb:.2f} GB")
print(f"shards per node: {per_node:,.0f}")
print("oversharded:", oversharded)
print("fix: consolidate into fewer, larger indices or delete cold shards")

assert average_gb == 0.8
assert per_node == 24_000
assert oversharded

### Try it 5. Size a bulk request from document size

**Task.** Documents average 2 KB, and the target request body is 5 MB. Compute the documents per request and the number of requests needed to load 50 million documents.

**Hint.** `docs_per_request = floor(target_bytes / doc_bytes)`; `requests = ceil(total / docs_per_request)`.

**Expected result.** 2,560 documents per request and 19,532 requests.

In [ ]:
import math

DOC_BYTES = 2048
TARGET_MB = 5
TOTAL_DOCS = 50_000_000

docs_per_request = (TARGET_MB * 1024 * 1024) // DOC_BYTES
requests = math.ceil(TOTAL_DOCS / docs_per_request)

print(f"documents per bulk request: {docs_per_request:,}")
print(f"bulk requests for {TOTAL_DOCS:,} docs: {requests:,}")

assert docs_per_request == 2560
assert requests == 19532

### Try it 6. Estimate aggregation memory from cardinality

**Task.** A `cardinality` aggregation uses `precision_threshold = 40000` and the query hits 10 shards. HyperLogLog++ uses roughly `precision_threshold * 8` bytes per shard. Compute the per-shard and total memory, and compare with the default threshold of 3000.

**Hint.** `per_shard = threshold * 8`; `total = per_shard * shards`.

**Expected result.** 320,000 bytes (312.5 KiB) per shard, 3,200,000 bytes total, versus 24,000 bytes per shard at the default.

In [ ]:
PRECISION_THRESHOLD = 40_000
DEFAULT_THRESHOLD = 3_000
SHARDS = 10
BYTES_PER_PRECISION = 8

per_shard_bytes = PRECISION_THRESHOLD * BYTES_PER_PRECISION
total_bytes = per_shard_bytes * SHARDS
default_per_shard_bytes = DEFAULT_THRESHOLD * BYTES_PER_PRECISION

print(f"per shard: {per_shard_bytes:,} bytes ({per_shard_bytes / 1024:.1f} KiB)")
print(f"total across {SHARDS} shards: {total_bytes:,} bytes")
print(f"default per shard: {default_per_shard_bytes:,} bytes")

assert per_shard_bytes == 320_000
assert total_bytes == 3_200_000
assert default_per_shard_bytes == 24_000

### Try it 7. Plan an ILM rollover from daily index size

**Task.** Logs arrive at 600 GB/day of primary data. Target 50 GB per shard, keep 90 days, and use one replica. Compute shards per day, total shards including replicas, and the on-disk footprint.

**Hint.** `shards_per_day = ceil(daily_gb / target_gb)`; multiply by retention and `(1 + replicas)` for the shard total.

**Expected result.** 12 shards per day, 2,160 shards total, 105.5 TB on disk.

In [ ]:
PRIMARY_GB_PER_DAY = 600
TARGET_SHARD_GB = 50
RETENTION_DAYS = 90
REPLICAS = 1

shards_per_day = -(-PRIMARY_GB_PER_DAY // TARGET_SHARD_GB)
total_shards = shards_per_day * RETENTION_DAYS * (1 + REPLICAS)
footprint_tb = PRIMARY_GB_PER_DAY * RETENTION_DAYS * (1 + REPLICAS) / 1024

print(f"shards per day: {shards_per_day}")
print(f"total shards with replicas: {total_shards:,}")
print(f"on-disk footprint: {footprint_tb:.1f} TB")
print("roll over on max_primary_shard_size so quiet days make fewer shards")

assert shards_per_day == 12
assert total_shards == 2160
assert round(footprint_tb, 1) == 105.5

### Try it 8. Compute deep pagination cost per shard

**Task.** A search pages `from = 9000, size = 100` across 20 shards. Compute the hits each shard must sort, the total fetched, and whether the page is inside `max_result_window = 10000`. Then compute the same for `search_after` with `size = 100`.

**Hint.** `per_shard = from + size`; `total = shards * per_shard`; a page is allowed while `from + size <= max_result_window`.

**Expected result.** 9,100 hits per shard, 182,000 total, inside the window, and 2,000 hits with `search_after`.

In [ ]:
SHARDS = 20
FROM = 9000
SIZE = 100
MAX_RESULT_WINDOW = 10000

per_shard = FROM + SIZE
total_fetched = SHARDS * per_shard
within_window = per_shard <= MAX_RESULT_WINDOW
search_after_fetched = SHARDS * SIZE

print(f"hits sorted per shard: {per_shard:,}")
print(f"hits fetched across {SHARDS} shards: {total_fetched:,}")
print("within max_result_window:", within_window)
print(f"hits with search_after: {search_after_fetched:,}")

assert per_shard == 9100
assert total_fetched == 182_000
assert within_window
assert search_after_fetched == 2000